# G37 — Frames for the motivation / case-study figures (no model, no training)

Candidates come from `analysis/case_study_select.py` on the **saved** test predictions (final model vs baseline;
descriptive use decided by the authors; final model = preregistered RoleNet, G10). This notebook keeps only MCIS where **at least three different people**
appear in clips I–III (ArcFace identities clustered as in RoleNet) and at least two in clip III, then renders:

* a one-row strip per MCIS (one frame per clip, chosen where most faces are visible) with face boxes coloured by
  role — **A** (speaker of clip III), **L** (listener = responder proxy), **O** (others) — and the transcripts;
* a 4 × 3 grid (three frames per clip) for checking.

People are identities seen in at least two sampled frames. A `_paper` version (no header, role legend) is saved
for the figure. Clip IV is shown only to illustrate the target turn; no model reads it. Inputs: `hi-ef-dataset`, `g8a-features`. CPU.

In [ ]:
import os, glob, pickle, textwrap
from collections import defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import AgglomerativeClustering

DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
G8A_GLOBS = ["/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features"]
OUT_DIR = "/kaggle/working/g37_frames"
SAME_PERSON_COS = 0.45
MIN_PEOPLE, MIN_PEOPLE_III = 3, 2
N_RENDER = {'win_shift': 4, 'win_mirror': 3, 'fail_shift_rare': 3}
FORCE_IDS = ['sample01305']                     # always rendered (motivation candidate chosen by the authors)
ONLY_FORCED = True                              # True: render only FORCE_IDS (fast); False: also the automatic picks
TEXT_OVERRIDE = {                               # subtitles with the punctuation the annotation file drops (paper only)
    'sample01305': ["You'd be a valuable asset to us.", "Claire should've never let you go. \u2013 Yeah.",
                    "Well\u2026 I'll hold my tongue.", "She and I don't talk."]}
CAND = [{'sample_id': 'sample00167', 'category': 'fail_shift_rare', 'clip1': '02/00090', 'clip2': '02/00091', 'clip3': '02/00092', 'clip4': '02/00093', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.063, 'p_true_Baseline': 0.158, 'certainty_IV': '2', 'margin': -0.095}, {'sample_id': 'sample00168', 'category': 'fail_shift_rare', 'clip1': '02/00096', 'clip2': '02/00097', 'clip3': '02/00098', 'clip4': '02/00099', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.046, 'p_true_Baseline': 0.084, 'certainty_IV': '3', 'margin': -0.038}, {'sample_id': 'sample00171', 'category': 'win_shift', 'clip1': '02/00108', 'clip2': '02/00109', 'clip3': '02/00110', 'clip4': '02/00111', 'A_III': 'sad', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.387, 'p_true_Baseline': 0.254, 'certainty_IV': '1', 'margin': 0.133}, {'sample_id': 'sample00184', 'category': 'fail_shift_rare', 'clip1': '02/00246', 'clip2': '02/00247', 'clip3': '02/00248', 'clip4': '02/00249', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.067, 'p_true_Baseline': 0.162, 'certainty_IV': '3', 'margin': -0.095}, {'sample_id': 'sample00189', 'category': 'fail_shift_rare', 'clip1': '02/00266', 'clip2': '02/00267', 'clip3': '02/00268', 'clip4': '02/00269', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.207, 'p_true_Baseline': 0.19, 'certainty_IV': '3', 'margin': 0.017}, {'sample_id': 'sample00191', 'category': 'fail_shift_rare', 'clip1': '02/00290', 'clip2': '02/00291', 'clip3': '02/00292', 'clip4': '02/00293', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.238, 'p_true_Baseline': 0.099, 'certainty_IV': '1', 'margin': 0.139}, {'sample_id': 'sample00198', 'category': 'win_shift', 'clip1': '02/00356', 'clip2': '02/00357', 'clip3': '02/00358', 'clip4': '02/00359', 'A_III': 'angry', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.267, 'p_true_Baseline': 0.132, 'certainty_IV': '1', 'margin': 0.135}, {'sample_id': 'sample00199', 'category': 'fail_shift_rare', 'clip1': '02/00357', 'clip2': '02/00358', 'clip3': '02/00359', 'clip4': '02/00360', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.087, 'p_true_Baseline': 0.119, 'certainty_IV': '1', 'margin': -0.032}, {'sample_id': 'sample00200', 'category': 'fail_shift_rare', 'clip1': '02/00383', 'clip2': '02/00384', 'clip3': '02/00385', 'clip4': '02/00386', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.089, 'p_true_Baseline': 0.097, 'certainty_IV': '1', 'margin': -0.008}, {'sample_id': 'sample00202', 'category': 'win_mirror', 'clip1': '02/00395', 'clip2': '02/00396', 'clip3': '02/00397', 'clip4': '02/00398', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.455, 'p_true_Baseline': 0.191, 'certainty_IV': '1', 'margin': 0.264}, {'sample_id': 'sample00203', 'category': 'fail_shift_rare', 'clip1': '02/00422', 'clip2': '02/00423', 'clip3': '02/00424', 'clip4': '02/00425', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.165, 'p_true_Baseline': 0.151, 'certainty_IV': '3', 'margin': 0.014}, {'sample_id': 'sample00210', 'category': 'fail_shift_rare', 'clip1': '02/00500', 'clip2': '02/00501', 'clip3': '02/00502', 'clip4': '02/00503', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.126, 'p_true_Baseline': 0.121, 'certainty_IV': '3', 'margin': 0.005}, {'sample_id': 'sample00213', 'category': 'fail_shift_rare', 'clip1': '02/00509', 'clip2': '02/00510', 'clip3': '02/00511', 'clip4': '02/00512', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.165, 'p_true_Baseline': 0.2, 'certainty_IV': '3', 'margin': -0.035}, {'sample_id': 'sample00214', 'category': 'fail_shift_rare', 'clip1': '02/00512', 'clip2': '02/00513', 'clip3': '02/00514', 'clip4': '02/00515', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.155, 'p_true_Baseline': 0.157, 'certainty_IV': '3', 'margin': -0.002}, {'sample_id': 'sample00219', 'category': 'win_mirror', 'clip1': '02/00617', 'clip2': '02/00618', 'clip3': '02/00619', 'clip4': '02/00620', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.302, 'p_true_Baseline': 0.085, 'certainty_IV': '1', 'margin': 0.217}, {'sample_id': 'sample00220', 'category': 'win_mirror', 'clip1': '02/00618', 'clip2': '02/00619', 'clip3': '02/00620', 'clip4': '02/00621', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.287, 'p_true_Baseline': 0.18, 'certainty_IV': '1', 'margin': 0.107}, {'sample_id': 'sample00225', 'category': 'win_shift', 'clip1': '02/00647', 'clip2': '02/00648', 'clip3': '02/00649', 'clip4': '02/00650', 'A_III': 'surprise', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.529, 'p_true_Baseline': 0.153, 'certainty_IV': '1', 'margin': 0.376}, {'sample_id': 'sample00227', 'category': 'fail_shift_rare', 'clip1': '02/00650', 'clip2': '02/00651', 'clip3': '02/00652', 'clip4': '02/00653', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.073, 'p_true_Baseline': 0.066, 'certainty_IV': '3', 'margin': 0.007}, {'sample_id': 'sample00231', 'category': 'win_mirror', 'clip1': '02/00678', 'clip2': '02/00679', 'clip3': '02/00680', 'clip4': '02/00681', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.314, 'p_true_Baseline': 0.2, 'certainty_IV': '1', 'margin': 0.114}, {'sample_id': 'sample00792', 'category': 'fail_shift_rare', 'clip1': '11/00080', 'clip2': '11/00081', 'clip3': '11/00082', 'clip4': '11/00083', 'A_III': 'sad', 'B_IV': 'fear', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.011, 'p_true_Baseline': 0.031, 'certainty_IV': '1', 'margin': -0.02}, {'sample_id': 'sample00795', 'category': 'win_shift', 'clip1': '11/00156', 'clip2': '11/00157', 'clip3': '11/00158', 'clip4': '11/00159', 'A_III': 'happy', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.425, 'p_true_Baseline': 0.204, 'certainty_IV': '1', 'margin': 0.221}, {'sample_id': 'sample00808', 'category': 'fail_shift_rare', 'clip1': '11/00246', 'clip2': '11/00247', 'clip3': '11/00248', 'clip4': '11/00249', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.143, 'p_true_Baseline': 0.197, 'certainty_IV': '1', 'margin': -0.054}, {'sample_id': 'sample00812', 'category': 'fail_shift_rare', 'clip1': '11/00263', 'clip2': '11/00264', 'clip3': '11/00265', 'clip4': '11/00266', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.064, 'p_true_Baseline': 0.12, 'certainty_IV': '1', 'margin': -0.056}, {'sample_id': 'sample01008', 'category': 'win_mirror', 'clip1': '15/00164', 'clip2': '15/00165', 'clip3': '15/00166', 'clip4': '15/00167', 'A_III': 'happy', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.316, 'p_true_Baseline': 0.22, 'certainty_IV': '1', 'margin': 0.096}, {'sample_id': 'sample01016', 'category': 'win_mirror', 'clip1': '15/00379', 'clip2': '15/00380', 'clip3': '15/00381', 'clip4': '15/00382', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'neutral', 'p_true_RoleNet': 0.361, 'p_true_Baseline': 0.171, 'certainty_IV': '1', 'margin': 0.19}, {'sample_id': 'sample01017', 'category': 'fail_shift_rare', 'clip1': '15/00387', 'clip2': '15/00388', 'clip3': '15/00389', 'clip4': '15/00390', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.227, 'p_true_Baseline': 0.141, 'certainty_IV': '1', 'margin': 0.086}, {'sample_id': 'sample01026', 'category': 'win_shift', 'clip1': '15/00521', 'clip2': '15/00522', 'clip3': '15/00523', 'clip4': '15/00524', 'A_III': 'sad', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.297, 'p_true_Baseline': 0.231, 'certainty_IV': '1', 'margin': 0.066}, {'sample_id': 'sample01028', 'category': 'fail_shift_rare', 'clip1': '15/00581', 'clip2': '15/00582', 'clip3': '15/00583', 'clip4': '15/00584', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'neutral', 'p_true_RoleNet': 0.094, 'p_true_Baseline': 0.126, 'certainty_IV': '1', 'margin': -0.032}, {'sample_id': 'sample01031', 'category': 'fail_shift_rare', 'clip1': '15/00639', 'clip2': '15/00640', 'clip3': '15/00641', 'clip4': '15/00642', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.264, 'p_true_Baseline': 0.154, 'certainty_IV': '1', 'margin': 0.11}, {'sample_id': 'sample01032', 'category': 'fail_shift_rare', 'clip1': '15/00677', 'clip2': '15/00678', 'clip3': '15/00679', 'clip4': '15/00680', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.062, 'p_true_Baseline': 0.066, 'certainty_IV': '1', 'margin': -0.004}, {'sample_id': 'sample01033', 'category': 'fail_shift_rare', 'clip1': '15/00682', 'clip2': '15/00683', 'clip3': '15/00684', 'clip4': '15/00685', 'A_III': 'sad', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.151, 'p_true_Baseline': 0.06, 'certainty_IV': '1', 'margin': 0.091}, {'sample_id': 'sample01035', 'category': 'fail_shift_rare', 'clip1': '15/00684', 'clip2': '15/00685', 'clip3': '15/00686', 'clip4': '15/00687', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.07, 'p_true_Baseline': 0.112, 'certainty_IV': '1', 'margin': -0.042}, {'sample_id': 'sample01038', 'category': 'win_shift', 'clip1': '15/00698', 'clip2': '15/00699', 'clip3': '15/00700', 'clip4': '15/00701', 'A_III': 'surprise', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.495, 'p_true_Baseline': 0.247, 'certainty_IV': '1', 'margin': 0.248}, {'sample_id': 'sample01041', 'category': 'win_shift', 'clip1': '15/00767', 'clip2': '15/00768', 'clip3': '15/00769', 'clip4': '15/00770', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'disgust', 'Baseline': 'angry', 'p_true_RoleNet': 0.259, 'p_true_Baseline': 0.149, 'certainty_IV': '1', 'margin': 0.11}, {'sample_id': 'sample01042', 'category': 'fail_shift_rare', 'clip1': '15/00775', 'clip2': '15/00776', 'clip3': '15/00777', 'clip4': '15/00778', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.087, 'p_true_Baseline': 0.092, 'certainty_IV': '1', 'margin': -0.005}, {'sample_id': 'sample01043', 'category': 'fail_shift_rare', 'clip1': '15/00790', 'clip2': '15/00791', 'clip3': '15/00792', 'clip4': '15/00793', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.154, 'p_true_Baseline': 0.051, 'certainty_IV': '1', 'margin': 0.103}, {'sample_id': 'sample01044', 'category': 'win_shift', 'clip1': '15/00811', 'clip2': '15/00812', 'clip3': '15/00813', 'clip4': '15/00814', 'A_III': 'sad', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.352, 'p_true_Baseline': 0.083, 'certainty_IV': '1', 'margin': 0.269}, {'sample_id': 'sample01047', 'category': 'fail_shift_rare', 'clip1': '16/00086', 'clip2': '16/00087', 'clip3': '16/00088', 'clip4': '16/00089', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.105, 'p_true_Baseline': 0.124, 'certainty_IV': '1', 'margin': -0.019}, {'sample_id': 'sample01048', 'category': 'win_mirror', 'clip1': '16/00097', 'clip2': '16/00098', 'clip3': '16/00099', 'clip4': '16/00100', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.381, 'p_true_Baseline': 0.065, 'certainty_IV': '2', 'margin': 0.316}, {'sample_id': 'sample01051', 'category': 'win_shift', 'clip1': '16/00115', 'clip2': '16/00116', 'clip3': '16/00117', 'clip4': '16/00118', 'A_III': 'sad', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.343, 'p_true_Baseline': 0.072, 'certainty_IV': '1', 'margin': 0.271}, {'sample_id': 'sample01059', 'category': 'fail_shift_rare', 'clip1': '16/00255', 'clip2': '16/00256', 'clip3': '16/00257', 'clip4': '16/00258', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.044, 'p_true_Baseline': 0.043, 'certainty_IV': '1', 'margin': 0.001}, {'sample_id': 'sample01060', 'category': 'fail_shift_rare', 'clip1': '16/00376', 'clip2': '16/00377', 'clip3': '16/00378', 'clip4': '16/00379', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.166, 'p_true_Baseline': 0.133, 'certainty_IV': '1', 'margin': 0.033}, {'sample_id': 'sample01075', 'category': 'fail_shift_rare', 'clip1': '16/00567', 'clip2': '16/00568', 'clip3': '16/00569', 'clip4': '16/00570', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'neutral', 'p_true_RoleNet': 0.078, 'p_true_Baseline': 0.102, 'certainty_IV': '3', 'margin': -0.024}, {'sample_id': 'sample01076', 'category': 'win_shift', 'clip1': '16/00568', 'clip2': '16/00569', 'clip3': '16/00570', 'clip4': '16/00571', 'A_III': 'disgust', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'neutral', 'p_true_RoleNet': 0.253, 'p_true_Baseline': 0.122, 'certainty_IV': '3', 'margin': 0.131}, {'sample_id': 'sample01077', 'category': 'fail_shift_rare', 'clip1': '16/00569', 'clip2': '16/00570', 'clip3': '16/00571', 'clip4': '16/00572', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'neutral', 'p_true_RoleNet': 0.153, 'p_true_Baseline': 0.136, 'certainty_IV': '1', 'margin': 0.017}, {'sample_id': 'sample01080', 'category': 'win_shift', 'clip1': '16/00616', 'clip2': '16/00617', 'clip3': '16/00618', 'clip4': '16/00619', 'A_III': 'angry', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.377, 'p_true_Baseline': 0.089, 'certainty_IV': '3', 'margin': 0.288}, {'sample_id': 'sample01081', 'category': 'win_shift', 'clip1': '16/00623', 'clip2': '16/00624', 'clip3': '16/00625', 'clip4': '16/00626', 'A_III': 'angry', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.552, 'p_true_Baseline': 0.116, 'certainty_IV': '3', 'margin': 0.436}, {'sample_id': 'sample01082', 'category': 'fail_shift_rare', 'clip1': '16/00624', 'clip2': '16/00625', 'clip3': '16/00626', 'clip4': '16/00627', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.097, 'p_true_Baseline': 0.165, 'certainty_IV': '1', 'margin': -0.068}, {'sample_id': 'sample01087', 'category': 'win_mirror', 'clip1': '16/00730', 'clip2': '16/00731', 'clip3': '16/00732', 'clip4': '16/00733', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.377, 'p_true_Baseline': 0.268, 'certainty_IV': '1', 'margin': 0.109}, {'sample_id': 'sample01092', 'category': 'fail_shift_rare', 'clip1': '16/00898', 'clip2': '16/00899', 'clip3': '16/00900', 'clip4': '16/00901', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.115, 'p_true_Baseline': 0.126, 'certainty_IV': '1', 'margin': -0.011}, {'sample_id': 'sample01095', 'category': 'fail_shift_rare', 'clip1': '16/00911', 'clip2': '16/00912', 'clip3': '16/00913', 'clip4': '16/00914', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.094, 'p_true_Baseline': 0.136, 'certainty_IV': '3', 'margin': -0.042}, {'sample_id': 'sample01096', 'category': 'win_shift', 'clip1': '16/00914', 'clip2': '16/00915', 'clip3': '16/00916', 'clip4': '16/00917', 'A_III': 'disgust', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.329, 'p_true_Baseline': 0.17, 'certainty_IV': '1', 'margin': 0.159}, {'sample_id': 'sample01305', 'category': 'win_mirror', 'clip1': '21/00047', 'clip2': '21/00048', 'clip3': '21/00049', 'clip4': '21/00050', 'A_III': 'happy', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'neutral', 'p_true_RoleNet': 0.588, 'p_true_Baseline': 0.139, 'certainty_IV': '1', 'margin': 0.449}, {'sample_id': 'sample01306', 'category': 'fail_shift_rare', 'clip1': '21/00049', 'clip2': '21/00050', 'clip3': '21/00051', 'clip4': '21/00052', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.028, 'p_true_Baseline': 0.114, 'certainty_IV': '1', 'margin': -0.086}, {'sample_id': 'sample01311', 'category': 'win_mirror', 'clip1': '21/00075', 'clip2': '21/00076', 'clip3': '21/00077', 'clip4': '21/00078', 'A_III': 'sad', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.423, 'p_true_Baseline': 0.191, 'certainty_IV': '2', 'margin': 0.232}, {'sample_id': 'sample01312', 'category': 'win_mirror', 'clip1': '21/00078', 'clip2': '21/00079', 'clip3': '21/00080', 'clip4': '21/00081', 'A_III': 'sad', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.493, 'p_true_Baseline': 0.251, 'certainty_IV': '1', 'margin': 0.242}, {'sample_id': 'sample01315', 'category': 'win_shift', 'clip1': '21/00143', 'clip2': '21/00144', 'clip3': '21/00145', 'clip4': '21/00146', 'A_III': 'happy', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.387, 'p_true_Baseline': 0.167, 'certainty_IV': '3', 'margin': 0.22}, {'sample_id': 'sample01316', 'category': 'fail_shift_rare', 'clip1': '21/00163', 'clip2': '21/00164', 'clip3': '21/00165', 'clip4': '21/00166', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'sad', 'p_true_RoleNet': 0.052, 'p_true_Baseline': 0.048, 'certainty_IV': '1', 'margin': 0.004}, {'sample_id': 'sample01318', 'category': 'fail_shift_rare', 'clip1': '21/00167', 'clip2': '21/00168', 'clip3': '21/00169', 'clip4': '21/00170', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.049, 'p_true_Baseline': 0.058, 'certainty_IV': '1', 'margin': -0.009}, {'sample_id': 'sample01338', 'category': 'fail_shift_rare', 'clip1': '21/00315', 'clip2': '21/00316', 'clip3': '21/00317', 'clip4': '21/00318', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.138, 'p_true_Baseline': 0.159, 'certainty_IV': '1', 'margin': -0.021}, {'sample_id': 'sample01340', 'category': 'fail_shift_rare', 'clip1': '21/00491', 'clip2': '21/00492', 'clip3': '21/00493', 'clip4': '21/00494', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.092, 'p_true_Baseline': 0.06, 'certainty_IV': '3', 'margin': 0.032}, {'sample_id': 'sample01342', 'category': 'win_mirror', 'clip1': '21/00525', 'clip2': '21/00526', 'clip3': '21/00527', 'clip4': '21/00528', 'A_III': 'neutral', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.237, 'p_true_Baseline': 0.143, 'certainty_IV': '1', 'margin': 0.094}, {'sample_id': 'sample01344', 'category': 'win_mirror', 'clip1': '21/00558', 'clip2': '21/00559', 'clip3': '21/00560', 'clip4': '21/00561', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.444, 'p_true_Baseline': 0.198, 'certainty_IV': '1', 'margin': 0.246}, {'sample_id': 'sample01345', 'category': 'fail_shift_rare', 'clip1': '21/00569', 'clip2': '21/00570', 'clip3': '21/00571', 'clip4': '21/00572', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.108, 'p_true_Baseline': 0.131, 'certainty_IV': '1', 'margin': -0.023}, {'sample_id': 'sample01348', 'category': 'fail_shift_rare', 'clip1': '21/00596', 'clip2': '21/00597', 'clip3': '21/00598', 'clip4': '21/00599', 'A_III': 'happy', 'B_IV': 'fear', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.014, 'p_true_Baseline': 0.022, 'certainty_IV': '3', 'margin': -0.008}, {'sample_id': 'sample01352', 'category': 'win_shift', 'clip1': '21/00704', 'clip2': '21/00705', 'clip3': '21/00706', 'clip4': '21/00707', 'A_III': 'angry', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.317, 'p_true_Baseline': 0.157, 'certainty_IV': '1', 'margin': 0.16}, {'sample_id': 'sample01612', 'category': 'win_shift', 'clip1': '28/00124', 'clip2': '28/00125', 'clip3': '28/00126', 'clip4': '28/00127', 'A_III': 'angry', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.335, 'p_true_Baseline': 0.137, 'certainty_IV': '1', 'margin': 0.198}, {'sample_id': 'sample01614', 'category': 'win_shift', 'clip1': '28/00148', 'clip2': '28/00149', 'clip3': '28/00150', 'clip4': '28/00151', 'A_III': 'disgust', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.265, 'p_true_Baseline': 0.126, 'certainty_IV': '3', 'margin': 0.139}, {'sample_id': 'sample01616', 'category': 'fail_shift_rare', 'clip1': '28/00206', 'clip2': '28/00207', 'clip3': '28/00208', 'clip4': '28/00209', 'A_III': 'neutral', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.134, 'p_true_Baseline': 0.072, 'certainty_IV': '1', 'margin': 0.062}, {'sample_id': 'sample01617', 'category': 'fail_shift_rare', 'clip1': '28/00214', 'clip2': '28/00215', 'clip3': '28/00216', 'clip4': '28/00217', 'A_III': 'sad', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.172, 'p_true_Baseline': 0.145, 'certainty_IV': '3', 'margin': 0.027}, {'sample_id': 'sample01619', 'category': 'fail_shift_rare', 'clip1': '28/00233', 'clip2': '28/00234', 'clip3': '28/00235', 'clip4': '28/00236', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.047, 'p_true_Baseline': 0.121, 'certainty_IV': '1', 'margin': -0.074}, {'sample_id': 'sample01628', 'category': 'fail_shift_rare', 'clip1': '28/00344', 'clip2': '28/00345', 'clip3': '28/00346', 'clip4': '28/00347', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'neutral', 'Baseline': 'happy', 'p_true_RoleNet': 0.121, 'p_true_Baseline': 0.057, 'certainty_IV': '1', 'margin': 0.064}, {'sample_id': 'sample01629', 'category': 'fail_shift_rare', 'clip1': '28/00345', 'clip2': '28/00346', 'clip3': '28/00347', 'clip4': '28/00348', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.141, 'p_true_Baseline': 0.034, 'certainty_IV': '3', 'margin': 0.107}, {'sample_id': 'sample01632', 'category': 'fail_shift_rare', 'clip1': '28/00379', 'clip2': '28/00380', 'clip3': '28/00381', 'clip4': '28/00382', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.033, 'p_true_Baseline': 0.047, 'certainty_IV': '1', 'margin': -0.014}, {'sample_id': 'sample01635', 'category': 'win_shift', 'clip1': '28/00503', 'clip2': '28/00504', 'clip3': '28/00505', 'clip4': '28/00506', 'A_III': 'angry', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.254, 'p_true_Baseline': 0.229, 'certainty_IV': '3', 'margin': 0.025}, {'sample_id': 'sample01636', 'category': 'fail_shift_rare', 'clip1': '28/00542', 'clip2': '28/00543', 'clip3': '28/00544', 'clip4': '28/00545', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.131, 'p_true_Baseline': 0.13, 'certainty_IV': '1', 'margin': 0.001}, {'sample_id': 'sample02380', 'category': 'fail_shift_rare', 'clip1': '47/00050', 'clip2': '47/00051', 'clip3': '47/00052', 'clip4': '47/00053', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.065, 'p_true_Baseline': 0.095, 'certainty_IV': '2', 'margin': -0.03}, {'sample_id': 'sample02383', 'category': 'fail_shift_rare', 'clip1': '47/00102', 'clip2': '47/00103', 'clip3': '47/00104', 'clip4': '47/00105', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.05, 'p_true_Baseline': 0.058, 'certainty_IV': '3', 'margin': -0.008}, {'sample_id': 'sample02386', 'category': 'fail_shift_rare', 'clip1': '47/00106', 'clip2': '47/00107', 'clip3': '47/00108', 'clip4': '47/00109', 'A_III': 'sad', 'B_IV': 'disgust', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.079, 'p_true_Baseline': 0.099, 'certainty_IV': '1', 'margin': -0.02}, {'sample_id': 'sample02387', 'category': 'fail_shift_rare', 'clip1': '47/00110', 'clip2': '47/00111', 'clip3': '47/00112', 'clip4': '47/00113', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.121, 'p_true_Baseline': 0.106, 'certainty_IV': '1', 'margin': 0.015}, {'sample_id': 'sample02388', 'category': 'fail_shift_rare', 'clip1': '47/00134', 'clip2': '47/00135', 'clip3': '47/00136', 'clip4': '47/00137', 'A_III': 'sad', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.14, 'p_true_Baseline': 0.131, 'certainty_IV': '1', 'margin': 0.009}, {'sample_id': 'sample02389', 'category': 'fail_shift_rare', 'clip1': '47/00135', 'clip2': '47/00136', 'clip3': '47/00137', 'clip4': '47/00138', 'A_III': 'disgust', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.072, 'p_true_Baseline': 0.052, 'certainty_IV': '2', 'margin': 0.02}, {'sample_id': 'sample02392', 'category': 'fail_shift_rare', 'clip1': '47/00149', 'clip2': '47/00150', 'clip3': '47/00151', 'clip4': '47/00152', 'A_III': 'sad', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.125, 'p_true_Baseline': 0.096, 'certainty_IV': '2', 'margin': 0.029}, {'sample_id': 'sample02396', 'category': 'fail_shift_rare', 'clip1': '47/00313', 'clip2': '47/00314', 'clip3': '47/00315', 'clip4': '47/00316', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'sad', 'p_true_RoleNet': 0.078, 'p_true_Baseline': 0.051, 'certainty_IV': '3', 'margin': 0.027}, {'sample_id': 'sample02398', 'category': 'win_shift', 'clip1': '47/00378', 'clip2': '47/00379', 'clip3': '47/00380', 'clip4': '47/00381', 'A_III': 'angry', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.339, 'p_true_Baseline': 0.222, 'certainty_IV': '1', 'margin': 0.117}, {'sample_id': 'sample02400', 'category': 'fail_shift_rare', 'clip1': '47/00428', 'clip2': '47/00429', 'clip3': '47/00430', 'clip4': '47/00431', 'A_III': 'fear', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.052, 'p_true_Baseline': 0.051, 'certainty_IV': '1', 'margin': 0.001}, {'sample_id': 'sample02401', 'category': 'win_shift', 'clip1': '47/00429', 'clip2': '47/00430', 'clip3': '47/00431', 'clip4': '47/00432', 'A_III': 'surprise', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.499, 'p_true_Baseline': 0.271, 'certainty_IV': '3', 'margin': 0.228}, {'sample_id': 'sample02402', 'category': 'fail_shift_rare', 'clip1': '47/00430', 'clip2': '47/00431', 'clip3': '47/00432', 'clip4': '47/00433', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.036, 'p_true_Baseline': 0.05, 'certainty_IV': '3', 'margin': -0.014}, {'sample_id': 'sample02404', 'category': 'fail_shift_rare', 'clip1': '47/00469', 'clip2': '47/00470', 'clip3': '47/00471', 'clip4': '47/00472', 'A_III': 'angry', 'B_IV': 'fear', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.021, 'p_true_Baseline': 0.02, 'certainty_IV': '3', 'margin': 0.001}, {'sample_id': 'sample02407', 'category': 'fail_shift_rare', 'clip1': '47/00502', 'clip2': '47/00503', 'clip3': '47/00504', 'clip4': '47/00505', 'A_III': 'disgust', 'B_IV': 'fear', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.017, 'p_true_Baseline': 0.023, 'certainty_IV': '3', 'margin': -0.006}, {'sample_id': 'sample02412', 'category': 'fail_shift_rare', 'clip1': '47/00517', 'clip2': '47/00518', 'clip3': '47/00519', 'clip4': '47/00520', 'A_III': 'happy', 'B_IV': 'fear', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.015, 'p_true_Baseline': 0.022, 'certainty_IV': '3', 'margin': -0.007}, {'sample_id': 'sample02414', 'category': 'fail_shift_rare', 'clip1': '47/00521', 'clip2': '47/00522', 'clip3': '47/00523', 'clip4': '47/00524', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.19, 'p_true_Baseline': 0.125, 'certainty_IV': '1', 'margin': 0.065}, {'sample_id': 'sample02415', 'category': 'win_shift', 'clip1': '47/00522', 'clip2': '47/00523', 'clip3': '47/00524', 'clip4': '47/00525', 'A_III': 'disgust', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'angry', 'p_true_RoleNet': 0.305, 'p_true_Baseline': 0.211, 'certainty_IV': '1', 'margin': 0.094}, {'sample_id': 'sample02416', 'category': 'fail_shift_rare', 'clip1': '47/00532', 'clip2': '47/00533', 'clip3': '47/00534', 'clip4': '47/00535', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.166, 'p_true_Baseline': 0.104, 'certainty_IV': '1', 'margin': 0.062}, {'sample_id': 'sample02419', 'category': 'win_shift', 'clip1': '47/00603', 'clip2': '47/00604', 'clip3': '47/00605', 'clip4': '47/00606', 'A_III': 'sad', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.34, 'p_true_Baseline': 0.221, 'certainty_IV': '2', 'margin': 0.119}, {'sample_id': 'sample02420', 'category': 'fail_shift_rare', 'clip1': '47/00611', 'clip2': '47/00612', 'clip3': '47/00613', 'clip4': '47/00614', 'A_III': 'disgust', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.049, 'p_true_Baseline': 0.054, 'certainty_IV': '1', 'margin': -0.005}, {'sample_id': 'sample02424', 'category': 'win_shift', 'clip1': '47/00645', 'clip2': '47/00646', 'clip3': '47/00647', 'clip4': '47/00648', 'A_III': 'neutral', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.393, 'p_true_Baseline': 0.228, 'certainty_IV': '2', 'margin': 0.165}, {'sample_id': 'sample02430', 'category': 'fail_shift_rare', 'clip1': '47/00668', 'clip2': '47/00669', 'clip3': '47/00670', 'clip4': '47/00671', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.056, 'p_true_Baseline': 0.097, 'certainty_IV': '3', 'margin': -0.041}, {'sample_id': 'sample02746', 'category': 'fail_shift_rare', 'clip1': '53/00019', 'clip2': '53/00020', 'clip3': '53/00021', 'clip4': '53/00022', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.07, 'p_true_Baseline': 0.091, 'certainty_IV': '1', 'margin': -0.021}, {'sample_id': 'sample02748', 'category': 'fail_shift_rare', 'clip1': '53/00021', 'clip2': '53/00022', 'clip3': '53/00023', 'clip4': '53/00024', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.041, 'p_true_Baseline': 0.073, 'certainty_IV': '1', 'margin': -0.032}, {'sample_id': 'sample02749', 'category': 'win_shift', 'clip1': '53/00031', 'clip2': '53/00032', 'clip3': '53/00033', 'clip4': '53/00034', 'A_III': 'angry', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.231, 'p_true_Baseline': 0.208, 'certainty_IV': '1', 'margin': 0.023}, {'sample_id': 'sample02754', 'category': 'win_shift', 'clip1': '53/00106', 'clip2': '53/00107', 'clip3': '53/00108', 'clip4': '53/00109', 'A_III': 'angry', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.449, 'p_true_Baseline': 0.186, 'certainty_IV': '1', 'margin': 0.263}, {'sample_id': 'sample02755', 'category': 'fail_shift_rare', 'clip1': '53/00142', 'clip2': '53/00143', 'clip3': '53/00144', 'clip4': '53/00145', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.179, 'p_true_Baseline': 0.138, 'certainty_IV': '1', 'margin': 0.041}, {'sample_id': 'sample02762', 'category': 'fail_shift_rare', 'clip1': '53/00184', 'clip2': '53/00185', 'clip3': '53/00186', 'clip4': '53/00187', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.142, 'p_true_Baseline': 0.155, 'certainty_IV': '1', 'margin': -0.013}, {'sample_id': 'sample02765', 'category': 'fail_shift_rare', 'clip1': '53/00193', 'clip2': '53/00194', 'clip3': '53/00195', 'clip4': '53/00196', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.118, 'p_true_Baseline': 0.153, 'certainty_IV': '1', 'margin': -0.035}, {'sample_id': 'sample02771', 'category': 'win_shift', 'clip1': '53/00248', 'clip2': '53/00249', 'clip3': '53/00250', 'clip4': '53/00251', 'A_III': 'sad', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.265, 'p_true_Baseline': 0.224, 'certainty_IV': '1', 'margin': 0.041}, {'sample_id': 'sample02773', 'category': 'win_mirror', 'clip1': '53/00250', 'clip2': '53/00251', 'clip3': '53/00252', 'clip4': '53/00253', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.311, 'p_true_Baseline': 0.216, 'certainty_IV': '1', 'margin': 0.095}, {'sample_id': 'sample02774', 'category': 'win_mirror', 'clip1': '53/00251', 'clip2': '53/00252', 'clip3': '53/00253', 'clip4': '53/00254', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.322, 'p_true_Baseline': 0.156, 'certainty_IV': '1', 'margin': 0.166}, {'sample_id': 'sample02779', 'category': 'win_shift', 'clip1': '53/00293', 'clip2': '53/00294', 'clip3': '53/00295', 'clip4': '53/00296', 'A_III': 'happy', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.311, 'p_true_Baseline': 0.179, 'certainty_IV': '3', 'margin': 0.132}, {'sample_id': 'sample02780', 'category': 'win_mirror', 'clip1': '53/00295', 'clip2': '53/00296', 'clip3': '53/00297', 'clip4': '53/00298', 'A_III': 'sad', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.327, 'p_true_Baseline': 0.136, 'certainty_IV': '2', 'margin': 0.191}, {'sample_id': 'sample02786', 'category': 'fail_shift_rare', 'clip1': '53/00401', 'clip2': '53/00402', 'clip3': '53/00403', 'clip4': '53/00404', 'A_III': 'happy', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.051, 'p_true_Baseline': 0.092, 'certainty_IV': '3', 'margin': -0.041}, {'sample_id': 'sample02788', 'category': 'fail_shift_rare', 'clip1': '53/00432', 'clip2': '53/00433', 'clip3': '53/00434', 'clip4': '53/00435', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.141, 'p_true_Baseline': 0.027, 'certainty_IV': '2', 'margin': 0.114}, {'sample_id': 'sample02794', 'category': 'win_mirror', 'clip1': '53/00460', 'clip2': '53/00461', 'clip3': '53/00462', 'clip4': '53/00463', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.493, 'p_true_Baseline': 0.236, 'certainty_IV': '1', 'margin': 0.257}, {'sample_id': 'sample02795', 'category': 'win_mirror', 'clip1': '53/00462', 'clip2': '53/00463', 'clip3': '53/00464', 'clip4': '53/00465', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'sad', 'p_true_RoleNet': 0.533, 'p_true_Baseline': 0.261, 'certainty_IV': '1', 'margin': 0.272}, {'sample_id': 'sample02799', 'category': 'fail_shift_rare', 'clip1': '53/00481', 'clip2': '53/00482', 'clip3': '53/00483', 'clip4': '53/00484', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.051, 'p_true_Baseline': 0.047, 'certainty_IV': '2', 'margin': 0.004}, {'sample_id': 'sample02801', 'category': 'fail_shift_rare', 'clip1': '53/00483', 'clip2': '53/00484', 'clip3': '53/00485', 'clip4': '53/00486', 'A_III': 'neutral', 'B_IV': 'disgust', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.039, 'p_true_Baseline': 0.031, 'certainty_IV': '2', 'margin': 0.008}, {'sample_id': 'sample02804', 'category': 'fail_shift_rare', 'clip1': '53/00487', 'clip2': '53/00488', 'clip3': '53/00489', 'clip4': '53/00490', 'A_III': 'sad', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.032, 'p_true_Baseline': 0.042, 'certainty_IV': '1', 'margin': -0.01}, {'sample_id': 'sample02811', 'category': 'fail_shift_rare', 'clip1': '53/00505', 'clip2': '53/00506', 'clip3': '53/00507', 'clip4': '53/00508', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.037, 'p_true_Baseline': 0.058, 'certainty_IV': '1', 'margin': -0.021}, {'sample_id': 'sample02813', 'category': 'fail_shift_rare', 'clip1': '53/00514', 'clip2': '53/00515', 'clip3': '53/00516', 'clip4': '53/00517', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.031, 'p_true_Baseline': 0.04, 'certainty_IV': '1', 'margin': -0.009}, {'sample_id': 'sample02824', 'category': 'win_shift', 'clip1': '53/00634', 'clip2': '53/00635', 'clip3': '53/00636', 'clip4': '53/00637', 'A_III': 'happy', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.404, 'p_true_Baseline': 0.244, 'certainty_IV': '1', 'margin': 0.16}]
os.makedirs(OUT_DIR, exist_ok=True)
roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
ANNOT_CSV = [os.path.join(r, "annotation.csv") for r in roots if os.path.exists(os.path.join(r, "annotation.csv"))]
G8A_DIR = next((g for g in G8A_GLOBS if os.path.isdir(g)), None)
assert VIDEO_ROOTS and ANNOT_CSV, "attach hi-ef-dataset"
assert G8A_DIR, "attach g8a-features"
ann = pd.read_csv(ANNOT_CSV[0], header=None, dtype=str).set_index(0)
need = {c for s in CAND for c in (s['clip1'], s['clip2'], s['clip3'])}
G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    d = pickle.load(open(f, 'rb'))
    G8.update({k: v for k, v in d.items() if k in need})
print(f"candidates {len(CAND)} | G8a records found {len(G8)} / {len(need)} clips")

In [ ]:
def roles(s):
    # identities over clips I-III (as RoleNet), then A / L / O from frame counts in clip III
    cl = [s['clip1'], s['clip2'], s['clip3']]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8.get(c, {}).get('faces', [])))]
    if not items:
        return None
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32)
    lab = (AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                   distance_threshold=1 - SAME_PERSON_COS).fit_predict(E) if len(items) > 1 else np.zeros(1, int))
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    role = {p: ('A' if p == A else 'L' if p == L else 'O') for p in set(lab)}
    tot = defaultdict(int)
    for (k, p), fr in frames.items():
        tot[p] += len(fr)
    # people = identities seen in >= 2 sampled frames (single-frame clusters are mostly background or split faces)
    return {'n_people': sum(v >= 2 for v in tot.values()), 'n_people_III': sum(len(frames[(2, p)]) >= 2 for p in ids3),
            'has_L': L is not None, 'faces': [(k, j, role[p]) for (k, j), p in zip(items, lab)]}


info = {}
for s in CAND:
    r = roles(s)
    if r is not None:
        info[s['sample_id']] = r
keep = [s for s in CAND if s['sample_id'] in info and info[s['sample_id']]['n_people'] >= MIN_PEOPLE
        and info[s['sample_id']]['n_people_III'] >= MIN_PEOPLE_III and info[s['sample_id']]['has_L']]
print(f"MCIS with >= {MIN_PEOPLE} people in I-III and >= {MIN_PEOPLE_III} in III: {len(keep)} / {len(CAND)}")
PICK = [s for s in CAND if s['sample_id'] in FORCE_IDS and s['sample_id'] in info]
for cat, n in ({} if ONLY_FORCED else N_RENDER).items():
    sub = [s for s in keep if s['category'] == cat]
    sub = sorted(sub, key=lambda s: (str(s['certainty_IV']), -s['margin'] if cat != 'fail_shift_rare' else s['margin']))
    PICK += [x for x in sub[:n] if x not in PICK]
pd.DataFrame([{**s, **{k: v for k, v in info[s['sample_id']].items() if k != 'faces'}} for s in keep]).to_csv(
    f"{OUT_DIR}/candidates_multiparty.csv", index=False)
print("to render:", [(s['sample_id'], s['category'], info[s['sample_id']]['n_people']) for s in PICK])

In [ ]:
COL = {'A': '#eb6834', 'L': '#2a78d6', 'O': '#8a8984'}


def video_path(clip):
    ep, num = clip.split('/')
    for root in VIDEO_ROOTS:
        for ext in ('.mp4', '.avi', '.mkv', '.mov'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


def grab(clip, t):
    p = video_path(clip)
    if p is None:
        return np.zeros((180, 320, 3), np.uint8)
    cap = cv2.VideoCapture(p)
    cap.set(cv2.CAP_PROP_POS_MSEC, max(t, 0) * 1000)
    ok, fr = cap.read()
    if not ok:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0); ok, fr = cap.read()
    cap.release()
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if ok else np.zeros((180, 320, 3), np.uint8)


def duration(clip):
    p = video_path(clip)
    if p is None:
        return 1.0
    cap = cv2.VideoCapture(p)
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()
    return max(n / fps, 0.1)


def text(c):
    t = ann.at[c, 1] if c in ann.index else ''
    return t if isinstance(t, str) else ''


def best_frame(s, k):
    # time of the sampled frame with most faces in clip k (clips I-III); middle of the clip for clip IV
    clips = [s['clip1'], s['clip2'], s['clip3'], s['clip4']]
    if k == 3:
        return duration(clips[3]) / 2, []
    fs = G8[clips[k]]['faces']
    by = defaultdict(list)
    for (kk, j, r) in info[s['sample_id']]['faces']:
        if kk == k:
            by[fs[j]['frame']].append((fs[j], r))
    if not by:
        return duration(clips[k]) / 2, []
    # prefer frames that show the listener, then the speaker, then the most faces
    fr = max(by, key=lambda f: (any(r == 'L' for _, r in by[f]), any(r == 'A' for _, r in by[f]), len(by[f]),
                                -abs(by[f][0][0]['t'] - duration(clips[k]) / 2)))
    return by[fr][0][0]['t'], by[fr]


for s in PICK:
    clips = [s['clip1'], s['clip2'], s['clip3'], s['clip4']]
    names = ['I (context)', 'II (context)', f"III (A: {s['A_III']})", f"IV (target B: {s['B_IV']})"]
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.0))
    for k in range(4):
        t, faces = best_frame(s, k)
        img = grab(clips[k], t)
        H, W = img.shape[:2]
        ax[k].imshow(img); ax[k].axis('off')
        for d, r in faces:
            x1, y1, x2, y2 = d['box'] * np.array([W, H, W, H])
            ax[k].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=COL[r]))
            ax[k].text(x1, y1 - 4, r, color='white', fontsize=8, weight='bold',
                       bbox=dict(facecolor=COL[r], edgecolor='none', pad=1))
        ax[k].set_title(names[k] + "\n" + "\n".join(textwrap.wrap('"' + text(clips[k]) + '"', 40)[:2]), fontsize=8)
    fig.suptitle(f"{s['sample_id']} [{s['category']}, {info[s['sample_id']]['n_people']} people]  "
                 f"ours: {s['RoleNet']} (p_true {s['p_true_RoleNet']})   baseline: {s['Baseline']} (p_true {s['p_true_Baseline']})",
                 fontsize=9)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{s['sample_id']}_strip.png", dpi=200); plt.close(fig)
    # paper version: no header, neutral titles, role legend; clip III is split into the speaker shot and the
    # listener's reaction shot when no sampled frame shows both
    txt = TEXT_OVERRIDE.get(s['sample_id'], [text(c) for c in clips])
    panels = [(0, None, 'Clip I (context)', txt[0]), (1, None, 'Clip II (context)', txt[1])]
    fs3 = G8[clips[2]]['faces']
    by3 = defaultdict(set)
    for (kk, j, r) in info[s['sample_id']]['faces']:
        if kk == 2:
            by3[fs3[j]['frame']].add(r)
    if any({'A', 'L'} <= v for v in by3.values()) or not any('A' in v for v in by3.values()):
        panels.append((2, None, f"Clip III (speaker A: {s['A_III']})", txt[2]))
    else:
        panels += [(2, 'A', f"Clip III: speaker A ({s['A_III']})", txt[2]), (2, 'L', 'Clip III: listener L (reaction)', '')]
    panels.append((3, None, 'Clip IV (target: responder B)', txt[3]))

    def frame_with(k, want):
        if want is None:
            return best_frame(s, k)
        fs = G8[clips[k]]['faces']
        by = defaultdict(list)
        for (kk, j, r) in info[s['sample_id']]['faces']:
            if kk == k:
                by[fs[j]['frame']].append((fs[j], r))
        cand = [f for f in by if any(r == want for _, r in by[f])]
        f = max(cand, key=lambda f: max((d['box'][2] - d['box'][0]) for d, r in by[f] if r == want))
        return by[f][0][0]['t'], by[f]

    fig, ax = plt.subplots(1, len(panels), figsize=(3.25 * len(panels), 3.1))
    for a_, (k, want, title, tx) in zip(ax, panels):
        t, faces = frame_with(k, want)
        img = grab(clips[k], t)
        H, W = img.shape[:2]
        a_.imshow(img); a_.axis('off')
        for d, r in faces:
            x1, y1, x2, y2 = d['box'] * np.array([W, H, W, H])
            a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=COL[r]))
            a_.text(x1, y1 - 4, r, color='white', fontsize=8, weight='bold', bbox=dict(facecolor=COL[r], edgecolor='none', pad=1))
        a_.set_title(title + ("\n" + "\n".join(textwrap.wrap('\u201c' + tx + '\u201d', 38)[:2]) if tx else "\n"), fontsize=8)
    ok_ = lambda p_: '\u2713' if p_ == s['B_IV'] else '\u2717'
    fig.text(0.5, 0.13, f"Forecast for B (true: {s['B_IV']}):  RoleNet {s['RoleNet']} {ok_(s['RoleNet'])}    "
             f"baseline {s['Baseline']} {ok_(s['Baseline'])}", ha='center', fontsize=9)
    fig.legend(handles=[mpatches.Patch(color=COL['A'], label='A: speaker of clip III'),
                        mpatches.Patch(color=COL['L'], label='L: listener (responder proxy)'),
                        mpatches.Patch(color=COL['O'], label='O: others')],
               loc='lower center', ncol=3, frameon=False, fontsize=8)
    fig.tight_layout(rect=(0, 0.17, 1, 0.95)); fig.savefig(f"{OUT_DIR}/{s['sample_id']}_paper.png", dpi=300)
    fig.savefig(f"{OUT_DIR}/{s['sample_id']}_paper.pdf"); plt.close(fig)
    fig, ax = plt.subplots(3, 4, figsize=(13, 6.2))
    for k in range(4):
        dur = duration(clips[k])
        for j, q in enumerate((0.15, 0.5, 0.85)):
            ax[j, k].imshow(grab(clips[k], q * dur)); ax[j, k].axis('off')
        ax[0, k].set_title(names[k], fontsize=8)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{s['sample_id']}_grid.png", dpi=110); plt.close(fig)
    print("rendered", s['sample_id'], s['category'])
pd.DataFrame(PICK).to_csv(f"{OUT_DIR}/picked.csv", index=False)
print("saved to", OUT_DIR)